## Импорт библиотек

In [1]:
import pandas as pd
import seaborn as sns
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


%matplotlib inline

In [2]:
movies = pd.read_csv('movies_metadata.csv', low_memory=False)
ratings = pd.read_csv('ratings_small.csv', low_memory=False)
keywords = pd.read_csv('keywords.csv', low_memory=False)

In [3]:
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype('int')

In [4]:
movies.head(5)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45463 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45463 non-null  object 
 1   belongs_to_collection  4491 non-null   object 
 2   budget                 45463 non-null  object 
 3   genres                 45463 non-null  object 
 4   homepage               7779 non-null   object 
 5   id                     45463 non-null  int64  
 6   imdb_id                45446 non-null  object 
 7   original_language      45452 non-null  object 
 8   original_title         45463 non-null  object 
 9   overview               44509 non-null  object 
 10  popularity             45460 non-null  object 
 11  poster_path            45077 non-null  object 
 12  production_companies   45460 non-null  object 
 13  production_countries   45460 non-null  object 
 14  release_date           45376 non-null  object 
 15  revenue

In [6]:
movies = movies[['id','original_title', 'production_companies' ,'release_date', 'genres', 'overview', 'vote_average', 'vote_count']].copy()

In [7]:
movies['genres'] = movies['genres'].fillna('[]').apply(literal_eval).apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

In [8]:
movies['production_companies'] = movies['production_companies'].fillna('[]').apply(literal_eval).apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

In [9]:
movies.head()

,id,original_title,production_companies,release_date,genres,overview,vote_average,vote_count
0,862,Toy Story,[Pixar Animation Studios],1995-10-30,"[Animation, Comedy, Family]","Led by Woody, Andy's toys live happily in his ...",7.7,5415.0
1,8844,Jumanji,"[TriStar Pictures, Teitler Film, Interscope Co...",1995-12-15,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,6.9,2413.0
2,15602,Grumpier Old Men,"[Warner Bros., Lancaster Gate]",1995-12-22,"[Romance, Comedy]",A family wedding reignites the ancient feud be...,6.5,92.0
3,31357,Waiting to Exhale,[Twentieth Century Fox Film Corporation],1995-12-22,"[Comedy, Drama, Romance]","Cheated on, mistreated and stepped on, the wom...",6.1,34.0
4,11862,Father of the Bride Part II,"[Sandollar Productions, Touchstone Pictures]",1995-02-10,[Comedy],Just when George Banks has recovered from his ...,5.7,173.0


In [10]:
keywords['keywords'] = keywords['keywords'].fillna('[]').apply(literal_eval).apply(lambda x: [i['name'] for i in x] if isinstance(x, list) else [])

In [11]:
keywords.head()

,id,keywords
0,862,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,"[board game, disappearance, based on children'..."
2,15602,"[fishing, best friend, duringcreditsstinger, o..."
3,31357,"[based on novel, interracial relationship, sin..."
4,11862,"[baby, midlife crisis, confidence, aging, daug..."


In [12]:
df = movies.merge(keywords[['id', 'keywords']], on='id', how='left')

In [13]:
df.head()

,id,original_title,production_companies,release_date,genres,overview,vote_average,vote_count,keywords
0,862,Toy Story,[Pixar Animation Studios],1995-10-30,"[Animation, Comedy, Family]","Led by Woody, Andy's toys live happily in his ...",7.7,5415.0,"[jealousy, toy, boy, friendship, friends, riva..."
1,8844,Jumanji,"[TriStar Pictures, Teitler Film, Interscope Co...",1995-12-15,"[Adventure, Fantasy, Family]",When siblings Judy and Peter discover an encha...,6.9,2413.0,"[board game, disappearance, based on children'..."
2,15602,Grumpier Old Men,"[Warner Bros., Lancaster Gate]",1995-12-22,"[Romance, Comedy]",A family wedding reignites the ancient feud be...,6.5,92.0,"[fishing, best friend, duringcreditsstinger, o..."
3,31357,Waiting to Exhale,[Twentieth Century Fox Film Corporation],1995-12-22,"[Comedy, Drama, Romance]","Cheated on, mistreated and stepped on, the wom...",6.1,34.0,"[based on novel, interracial relationship, sin..."
4,11862,Father of the Bride Part II,"[Sandollar Productions, Touchstone Pictures]",1995-02-10,[Comedy],Just when George Banks has recovered from his ...,5.7,173.0,"[baby, midlife crisis, confidence, aging, daug..."


### 10 популярных фильмов на основе взвешенных рейтингов

In [14]:
def weighted_rating(x, m=df['vote_count'].quantile(0.8), C=df['vote_average'].mean()):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (m + v) * C)

df['weighted_rating'] = movies.apply(weighted_rating, axis=1)

In [15]:
top10_popular = df.sort_values('weighted_rating', ascending=False)['original_title'].head(10).tolist()

In [16]:
top10_popular

['Teen Witch',
 'The Shawshank Redemption',
 'Wuya yu maque',
 'Unaware',
 'Justice League: The New Frontier',
 'Penitentiary',
 'Pulp Fiction',
 "Schindler's List",
 'Jennifer',
 'The Banger Sisters']

### 10 популярных фильмов по жанру

In [17]:
s = df.apply(lambda x: pd.Series(x['genres']),axis=1).stack().reset_index(level=1, drop=True)
s.name = 'genre'
gen_md = df.drop('genres', axis=1).join(s)

In [18]:
def recommend_by_genre(genre: str):
    df1 = gen_md[gen_md['genre'] == genre]

    vote_counts = df1[df1['vote_count'].notnull()]['vote_count'].astype('int')
    vote_averages = df1[df1['vote_average'].notnull()]['vote_average'].astype('int')
    C = vote_averages.mean()
    m = vote_counts.quantile(0.85)

    qualified = df1[(df1['vote_count'] >= m) & (df1['vote_count'].notnull()) & (df1['vote_average'].notnull())][['original_title', 'vote_count', 'vote_average']]
    qualified['vote_count'] = qualified['vote_count'].astype('int')
    qualified['vote_average'] = qualified['vote_average'].astype('int')

    qualified['wr'] = qualified.apply(lambda x: (x['vote_count']/(x['vote_count']+m) * x['vote_average']) + (m/(m+x['vote_count']) * C), axis=1)
    qualified = qualified.sort_values('wr', ascending=False).head(250)

    return qualified

In [19]:
top10_genres = recommend_by_genre('Adventure')['original_title'].head(10).tolist()

In [20]:
top10_genres

['Inception',
 'Interstellar',
 'The Lord of the Rings: The Fellowship of the Ring',
 'The Lord of the Rings: The Return of the King',
 'The Lord of the Rings: The Two Towers',
 'Star Wars',
 'Back to the Future',
 'The Empire Strikes Back',
 '千と千尋の神隠し',
 'ハウルの動く城']

### 10 популярных фильмов по названию

In [21]:
df['genres'] = df['genres'].apply(lambda x: ','.join(x) if isinstance(x, list) else str(x))

In [22]:
df['keywords'] = df['keywords'].apply(lambda x: ','.join(x) if isinstance(x, list) else str(x))

In [23]:
df['total'] = df['genres'] + ' ' + df['keywords'] + ' ' + df['overview']
df['total'] = df['total'].fillna('')

In [24]:
df['total']

0        Animation,Comedy,Family jealousy,toy,boy,frien...
1        Adventure,Fantasy,Family board game,disappeara...
2        Romance,Comedy fishing,best friend,duringcredi...
3        Comedy,Drama,Romance based on novel,interracia...
4        Comedy baby,midlife crisis,confidence,aging,da...
                               ...                        
46478    Drama,Family tragic love Rising and falling be...
46479    Drama artist,play,pinoy An artist struggles to...
46480    Action,Drama,Thriller  When one of her hits go...
46481      In a small town live two brothers, one a min...
46482      50 years after decriminalisation of homosexu...
Name: total, Length: 46483, dtype: object

In [25]:
tf = TfidfVectorizer(stop_words='english')
tf_matrix = tf.fit_transform(df['total'])

In [26]:

nn_model = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute')
nn_model.fit(tf_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=11)

In [27]:
titles = df['original_title']
title_indices = pd.Series(df.index, index=df['original_title'])

def recommend_by_name(name: str):
    idx = title_indices[name]
    movie_vector = tf_matrix[idx]
    distances, indices = nn_model.kneighbors(movie_vector, n_neighbors=11)
    movie_indices = indices[0][1:]
    return movies['original_title'].iloc[movie_indices].tolist()

In [28]:
top10_names = recommend_by_name('The Godfather')

In [29]:
top10_names

['Henry V',
 'El auge del humano',
 'Ask the Dust',
 'Sparks',
 'Triumph of the Nerds: The Rise of Accidental Empires',
 'Good Boy!',
 "L'Enquête",
 "Jane Austen's Mafia!",
 'Trial by Jury',
 '新龍門客棧']

### Коллаборативная фильтрация

In [30]:
ratings_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
movies_matrix = ratings_matrix.T

nn_model_cf = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model_cf.fit(movies_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [31]:
def collaborative_filtering(user_id: int, n: int = 10):
    user_ratings = ratings_matrix.loc[user_id]
    high_rated_movies = user_ratings[user_ratings >= 4].index


    scores = {}

    for movie_id in high_rated_movies:
        if movie_id in movies_matrix.index:
            movie_idx = movies_matrix.index.get_loc(movie_id)
            distances, indices = nn_model_cf.kneighbors(movies_matrix.iloc[movie_idx].values.reshape(1, -1), n_neighbors=21)
            sim_movies = [movies_matrix.index[i] for i in indices.flatten()[1:]]
            sim_distances = distances.flatten()[1:]

            for sim_movie, dist in zip(sim_movies, sim_distances):
                if sim_movie not in user_ratings.index or user_ratings[sim_movie] == 0:
                    similarity = 1 - dist
                    if sim_movie not in scores:
                        scores[sim_movie] = 0
                    scores[sim_movie] += similarity

    top_movies = sorted(scores, key=scores.get, reverse=True)[:n]

    top_titles = df[df['id'].isin(top_movies)]['original_title'].head(10).tolist()

    return top_titles

In [32]:
top10_cf = collaborative_filtering(20)
top10_cf

['A Nightmare on Elm Street',
 'Back to the Future Part II',
 'Dawn of the Dead',
 "Dave Chappelle's Block Party"]

## Сохранение модели

In [33]:
import pickle

In [34]:
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tf, f)

with open('nn_model.pkl', 'wb') as f:
    pickle.dump(nn_model, f)

with open('nn_model_cf.pkl', 'wb') as f:
    pickle.dump(nn_model_cf, f)

In [35]:
user_ids = ratings['userId'].unique()
with open('user_ids.pkl', 'wb') as f:
    pickle.dump(user_ids, f)